# AISD to nnUnet data format

# Importación de librerías y funciones auxiliares

In [ ]:
import SimpleITK as sitk
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
input_dir = "D:/jm.rivera/AISD/DICOM"
masks_dir = "D:/jm.rivera/AISD/mask"
output_dir = "D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset001_AISD/"
imagesTr = os.path.join(output_dir, "imagesTr")
labelsTr = os.path.join(output_dir, "labelsTr")

# DICOM to NIfTI

In [ ]:
os.makedirs(output_dir, exist_ok=True)
os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

In [ ]:
def read_dicom_series(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    return image

In [ ]:
patient_folders = sorted(os.listdir(input_dir))

for idx, patient in enumerate(patient_folders, start=1):

    patient_path = os.path.join(input_dir, patient)

    if not os.path.isdir(patient_path):
        continue

    print(f"Processing {patient}...")

    # Read CT
    ct_image = read_dicom_series(patient_path)
    print("Size:", ct_image.GetSize())
    print("Spacing:", ct_image.GetSpacing())

    # nnU-Net naming
    case_id = f"AISD_{idx:03d}"

    image_filename = os.path.join(imagesTr, f"{case_id}_0000.nii.gz")

    sitk.WriteImage(ct_image, image_filename)
    print(f'New id: {case_id}')
    print('--'*10)

print("Conversion finished.")


# JPEG masks to NIfTI

In [ ]:
def create_binary_mask_from_jpegs(mask_folder, reference_ct):

    mask_files = sorted([
        f for f in os.listdir(mask_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    ct_size = reference_ct.GetSize()  # (x, y, z)

    if len(mask_files) != ct_size[2]:
        raise ValueError("Mismatch in number of slices!")

    mask_slices = []

    for f in mask_files:
        img = Image.open(os.path.join(mask_folder, f)).convert("L")
        arr = np.array(img)

        if arr.shape != (ct_size[1], ct_size[0]):
            raise ValueError("Mask slice size does not match CT slice size!")

        print(f'Mask unique values: {np.unique(arr)}')
        arr[np.isin(arr, [1, 2, 3, 5])] = 1
        mask_slices.append(arr)

    mask_3d = np.stack(mask_slices, axis=0)

    mask_image = sitk.GetImageFromArray(mask_3d.astype(np.uint8))

    mask_image.CopyInformation(reference_ct)

    return mask_image


In [ ]:
patient_folders = sorted(os.listdir(input_dir))
for idx, patient in enumerate(patient_folders, start=1):

    patient_path = os.path.join(input_dir, patient)
    mask_path = os.path.join(masks_dir, patient)

    if not os.path.isdir(patient_path):
        continue

    print(f"Processing {patient}...")

    # Read CT
    ct_image = read_dicom_series(patient_path)

    mask_vol = create_binary_mask_from_jpegs(mask_path, ct_image)

    # nnU-Net naming
    case_id = f"AISD_{idx:03d}"

    label_filename = os.path.join(labelsTr, f"{case_id}.nii.gz")

    sitk.WriteImage(mask_vol, label_filename)
    print(f'New id: {case_id}')
    print('--'*10)

print("Conversion finished.")

# Visual confirmation of masks alignment

In [ ]:
def apply_window(image, wl, ww):
    lower = wl - ww / 2
    upper = wl + ww / 2
    windowed = np.clip(image, lower, upper)
    return windowed

In [ ]:
index = '330'

wl = 40
ww = 40

ct_image = sitk.ReadImage(os.path.join(imagesTr, f'AISD_{index}_0000.nii.gz'))
mask_image = sitk.ReadImage(os.path.join(labelsTr, f'AISD_{index}.nii.gz'))

ct_np = sitk.GetArrayFromImage(ct_image)      # (z,y,x)
mask_np = sitk.GetArrayFromImage(mask_image)  # (z,y,x)

# Find slices containing lesion
lesion_area_per_slice = mask_np.sum(axis=(1,2))
z = np.argmax(lesion_area_per_slice)
windowed_slice = apply_window(ct_np[z], wl, ww)

if lesion_area_per_slice[z] == 0:
    print("⚠️ Empty mask")
else:
    print(f"Showing slice {z} (largest lesion area)")
    
    plt.imshow(windowed_slice, cmap="gray")
    plt.imshow(mask_np[z], alpha=0.4)
    plt.title(f"Max lesion slice {z}")
    plt.show()